In [ ]:
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
%matplotlib inline
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
OUT = ROOT / 'results'
OUT.mkdir(exist_ok=True)
RAW = Path(os.environ.get('ASTRO320W_DATA', str(ROOT / 'raw')))

In [ ]:
import numpy as np
from scipy.optimize import least_squares
from astropy.io import fits
from pathlib import Path
import csv
root=RAW/'Exercise E/EXERCISE E'
def fit(im,x,y):
 xi,yi=int(round(x)),int(round(y));v=im[yi-8:yi+9,xi-8:xi+9].astype(float);yy,xx=np.mgrid[yi-8:yi+9,xi-8:xi+9];bg=np.median(v)
 def model(p):
  a,cx,cy,sx,sy,c,dx,dy=p
  return a*np.exp(-.5*(((xx-cx)/sx)**2+((yy-cy)/sy)**2))+c+dx*(xx-xi)+dy*(yy-yi)
 p=[max(v.max()-bg,1),x,y,2.5,2.5,bg,0,0]
 r=least_squares(lambda p:(model(p)-v).ravel(),p,bounds=([0,x-4,y-4,.8,.8,0,-100,-100],[1e6,x+4,y+4,8,8,1e5,100,100]),loss='soft_l1',f_scale=20)
 return r.x,np.sqrt(np.mean((model(r.x)-v)**2)),v.max()
rows=[]
for n in [18,19,20]:
 im=fits.getdata(root/f'CCD Image {n}.fit'); yy,xx=np.unravel_index(np.argmax(im[595:625,670:700]),(30,30));xp,yp=xx+670,yy+595
 A,er,peak=fit(im,xp,yp)
 # Bright primary: positive-background-subtracted centroid in a fixed 12-pixel radius.
 cx,cy=683.,609.
 for k in range(5):
  yy,xx=np.mgrid[int(cy)-16:int(cy)+17,int(cx)-16:int(cx)+17];v=im[yy,xx].astype(float);r=np.hypot(xx-cx,yy-cy);bg=np.median(v[r>13]);weights=np.maximum(v-bg,0)*(r<=12);cx=np.sum(xx*weights)/weights.sum();cy=np.sum(yy*weights)/weights.sum()
 A[1:3]=cx,cy
 B,err,pk=fit(im,A[1]+26.4,A[2]-57.6)
 rows.append([n,A[1],1528-A[2],B[1],1528-B[2],pk,err])
with open(OUT/'HR7310_centroids.csv','w') as f:
 w=csv.writer(f);w.writerow(['original_image','x1_IJ','y1_IJ','x2_IJ','y2_IJ','companion_peak','companion_fit_rms']);w.writerows(rows)


In [ ]:
import numpy as np
from scipy.optimize import least_squares
from astropy.io import fits
from pathlib import Path
import csv
root=RAW/'Exercise E/EXERCISE E'
def fit(im,x,y):
 xi,yi=int(round(x)),int(round(y));v=im[yi-8:yi+9,xi-8:xi+9].astype(float);yy,xx=np.mgrid[yi-8:yi+9,xi-8:xi+9];bg=np.median(v)
 def model(p):
  a,cx,cy,sx,sy,c,dx,dy=p
  return a*np.exp(-.5*(((xx-cx)/sx)**2+((yy-cy)/sy)**2))+c+dx*(xx-xi)+dy*(yy-yi)
 p=[max(v.max()-bg,1),x,y,2.5,2.5,bg,0,0]
 r=least_squares(lambda p:(model(p)-v).ravel(),p,bounds=([0,x-4,y-4,.8,.8,0,-100,-100],[1e6,x+4,y+4,8,8,1e5,100,100]),loss='soft_l1',f_scale=20)
 return r.x,np.sqrt(np.mean((model(r.x)-v)**2)),v.max()
rows=[]
# Frame 17 excluded: user reports a streak/image artifact.
for n in [15,16]:
 im=fits.getdata(root/f'CCD Image {n}.fit')
 A,er,peak=fit(im,1073.,822.)
 B,err,pk=fit(im,1063.,789.)
 
 rows.append([n,A[1],1528-A[2],B[1],1528-B[2],pk,err])
with open(OUT/'HR6446_centroids.csv','w') as f:
 w=csv.writer(f);w.writerow(['original_image','x1_IJ','y1_IJ','x2_IJ','y2_IJ','companion_peak','companion_fit_rms']);w.writerows(rows)